# PDF문서 전처리하기

In [2]:
!pip install pyMuPDF


[notice] A new release of pip is available: 23.2.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
import pymupdf
import os

#1
pdf_file_path = "./과정기반 작물모형을 이용한 웹 기반 밀 재배관리 의사결정 지원시스템 설계 및 구축.pdf"
doc = pymupdf.open(pdf_file_path)

full_text = ''

#2 doc객체에서 각 페이지를 반복하여 텍스트를 추출하고, 이를 full_text라는 변수에 한페이지씩 추가한다.
for page in doc: # 문서 페이지 반복
    text = page.get_text() # 페이지 텍스트 추출
    full_text += text

#3 PDF파일을 텍스트 파일로 저장하기 위해 원본 PDF파일의 이름을 추출한다.
pdf_file_name = os.path.basename(pdf_file_path)
pdf_file_name = os.path.splitext(pdf_file_name)[0] # 확장자 제거

#4 
txt_file_path = f"./{pdf_file_name}.txt"
with open(txt_file_path, 'w', encoding='utf-8') as f:
    f.write(full_text)

## 전처리 하기 
페이지 상단이나 하단에 부가 내용이 많은 문서는 GTP같은 언어 모델이 제대로 처리 하지 못한다.<br />
헤더와 푸터 영역을 제외하고 텍스트만 추출

In [7]:
import pymupdf
import os

pdf_file_path = "./과정기반 작물모형을 이용한 웹 기반 밀 재배관리 의사결정 지원시스템 설계 및 구축.pdf"
doc = pymupdf.open(pdf_file_path)

header_height = 80
footer_height = 80

full_text = ''

for page in doc:
    rect = page.rect # 페이지 크기 가져오기
    
    header = page.get_text(clip=(0, 0, rect.width , header_height))
    footer = page.get_text(clip=(0, rect.height - footer_height, rect.width , rect.height))
    text = page.get_text(clip=(0, header_height, rect.width , rect.height - footer_height))

    full_text += text + '\n------------------------------------\n'

# 파일명만 추출
pdf_file_name = os.path.basename(pdf_file_path)
pdf_file_name = os.path.splitext(pdf_file_name)[0] # 확장자 제거

txt_file_path = f'./{pdf_file_name}_with_preprocessing.txt'

with open(txt_file_path, 'w', encoding='utf-8') as f:
    f.write(full_text)

페이지 하단의 푸터 내용과 페이지 상단에 있던 페이지 번호와 논문 제목 등을 제외하고 본문 내용만 잘 입력되어 있음.

## Zero-shot Prompting를 이용한 논문을 요약해주는 AI연구원 만들기

In [ ]:
!pip install google-genai python-dotenv

In [ ]:
# summary.py
from google import genai
from google.genai import types
from dotenv import load_dotenv
import os
os.environ["GOOGLE_API_KEY"] = "your_api_key"

load_dotenv()

api_key = os.getenv("GOOGLE_API_KEY")

def summarize_txt(file_path: str):

    # Gemini 클라이언트 생성
    client = genai.Client(api_key=api_key)

    # 텍스트 파일 읽기
    with open(file_path, "r", encoding="utf-8") as f:
        txt = f.read()

    # System Prompt
    system_prompt = f"""
너는 다음 글을 요약하는 봇이다.

아래 글을 읽고, 저자의 문제 인식과 주장을 파악하고, 주요 내용을 요약하라.

작성해야 하는 포맷은 다음과 같다.

# 제목

## 저자의 문제 인식 및 주장 (15문장 이내)

## 저자 소개
"""

    print(system_prompt)
    print("=" * 50)

    # Gemini 호출
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=txt,
        config=types.GenerateContentConfig(
            temperature=0.1,
            system_instruction=system_prompt
        )
    )

    return response.text


if __name__ == "__main__":

    file_path = "./과정기반 작물모형을 이용한 웹 기반 밀 재배관리 의사결정 지원시스템 설계 및 구축_with_preprocessing.txt"

    summary = summarize_txt(file_path)

    print(summary)

    output_path = "./crop_model_summary.txt"

    with open(output_path, "w", encoding="utf-8") as f:
        f.write(summary)

    print(f"\n요약이 저장되었습니다.\n{output_path}")

In [ ]:
!pip install google-genai python-dotenv pymupdf

## Zero-shot Prompting를 이용한 pdf파일을 요약해주는 AI연구원 만들기

In [ ]:
from google import genai
from google.genai import types
from dotenv import load_dotenv
import os
os.environ["GOOGLE_API_KEY"] = "your_api_key"

import pymupdf

# .env 로드
load_dotenv()

api_key = os.getenv("GOOGLE_API_KEY")


def pdf_to_text(pdf_file_path: str):

    doc = pymupdf.open(pdf_file_path)

    header_height = 80
    footer_height = 80

    full_text = ""

    for page in doc:
        rect = page.rect

        # Header와 Footer를 제외한 본문 추출
        text = page.get_text(
            clip=(
                0,
                header_height,
                rect.width,
                rect.height - footer_height,
            )
        )

        full_text += text
        full_text += "\n------------------------------------\n"

    # 파일명 추출
    pdf_file_name = os.path.basename(pdf_file_path)
    pdf_file_name = os.path.splitext(pdf_file_name)[0]

    txt_file_path = f"./{pdf_file_name}_with_preprocessing.txt"

    with open(txt_file_path, "w", encoding="utf-8") as f:
        f.write(full_text)

    return txt_file_path


def summarize_txt(file_path: str):

    client = genai.Client(api_key=api_key)

    # 텍스트 읽기
    with open(file_path, "r", encoding="utf-8") as f:
        txt = f.read()

    # System Prompt
    system_prompt = """
너는 다음 글을 요약하는 봇이다.

아래 글을 읽고,
저자의 문제 인식과 주장을 파악하고,
주요 내용을 요약하라.

작성해야 하는 포맷은 다음과 같다.

# 제목

## 저자의 문제 인식 및 주장 (15문장 이내)

## 저자 소개
"""

    print(system_prompt)
    print("=" * 50)

    # Gemini 호출
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=txt,
        config=types.GenerateContentConfig(
            temperature=0.1,
            system_instruction=system_prompt,
        ),
    )

    return response.text


def summarize_pdf(pdf_file_path: str, output_file_path: str):

    txt_file_path = pdf_to_text(pdf_file_path)

    summary = summarize_txt(txt_file_path)

    with open(output_file_path, "w", encoding="utf-8") as f:
        f.write(summary)


if __name__ == "__main__":

    pdf_file_path = (
        "./과정기반 작물모형을 이용한 웹 기반 밀 재배관리 의사결정 지원시스템 설계 및 구축.pdf"
    )

    summarize_pdf(
        pdf_file_path,
        "./crop_model_summary2.txt",
    )

    print("요약 완료")